# Decision Reviewers with Jev

In this notebook, we will demonstrate LatteReview's **decision reviewers**, which use a System One decision model
such as TypeSafe's **Jev** instead of an LLM.

## Jev in two minutes

An LLM reviewer writes text: it reads a prompt and generates JSON with a score and a written reasoning. A decision
model does not write anything. It reads a text **state** (here, an article's title and abstract) and a set of typed
**questions**, and returns a probability for every allowed answer:

| Question type | You define | You get back |
|---|---|---|
| `Noul` | a yes/no question | the probability of "yes" |
| `Choice` | named options | the chosen option and a probability per option |
| `Score` | 2-10 ordered levels | the most likely level, a probability per level, and the expected level |

For literature screening, this means answers that can never be off-schema, probabilities you can threshold for a
target recall, and a cost of about $0.00006 per article (Jev bills $0.042 per million input tokens; output is free).
What you give up: there is no written reasoning, no images, no `model_args`/`examples`/`reasoning`, and the model is
optimized for English text of up to 32k tokens per item. Jev also does not do multi-hop reasoning, so criteria
that need several reasoning steps are better left to an LLM (see the hybrid tutorial).

## Setting up the notebook

High-level configs

In [1]:
from dotenv import load_dotenv

# Load environment variables from .env file. Adjust the path to the .env file as needed.
load_dotenv(dotenv_path="../../.env")

# Enable asyncio in Jupyter
import asyncio
import nest_asyncio

nest_asyncio.apply()

# Add the package to the path (required if you are running this notebook from the tutorials folder)
import sys

sys.path.append("../../")

Import required packages

In [2]:
import time

import pandas as pd

from lattereview.providers import SystemOneProvider, Noul, Choice, Score
from lattereview.agents import DecisionReviewer, DecisionTitleAbstractReviewer, DecisionScoringReviewer
from lattereview.workflows import ReviewWorkflow
from lattereview.utils import suggest_threshold

pd.set_option("display.max_colwidth", 70)

## Choosing a backend

`SystemOneProvider` speaks the `/v1/systemone` protocol, so every backend works the same way:

- **TypeSafe** (default): reads `TYPESAFE_API_KEY`; the model defaults to `jev-latest`.
- **OpenRouter**: reads `OPENROUTER_API_KEY`; the model defaults to `~typesafe/jev-latest`.
- **Any other server** via `base_url`, e.g., a self-hosted OpenJev server. Local servers usually need no key.

This notebook runs on TypeSafe. The other backends are shown commented out.

In [3]:
provider = SystemOneProvider(backend="typesafe")

# provider = SystemOneProvider(backend="openrouter")
# provider = SystemOneProvider(base_url="http://localhost:3000")  # a local OpenJev server (openjev serve ...)
provider

SystemOneProvider(provider='SystemOne', backend='typesafe', base_url=None, model='jev-latest', timeout=60.0, max_retries=3, requests_per_minute=1000, input_price_per_million=0.042, endpoint='https://api.typesafe.ai/v1/systemone')

Before using a reviewer, let's ask the provider a few questions about one abstract directly. All questions about a
state go in **one request**: the state is billed once, and each extra question adds only a few tokens.

In [4]:
state = (
    "Title: Deep learning for pneumonia detection on chest radiographs.\n"
    "Abstract: We trained a convolutional neural network on 100,000 chest X-rays from one hospital and validated it "
    "on an external cohort of 5,000 patients from another hospital."
)
questions = {
    "uses_deep_learning": Noul("Does the study use deep learning?"),
    "modality": Choice("Which imaging modality is studied?", ["CT", "MRI", "X-ray", "ultrasound"]),
    "validation": Score("How strong is the validation?", ["none", "internal only", "external"]),
}

result = asyncio.run(provider.decide(state, questions))
for question_id, answer in result.answers.items():
    print(f"{question_id}: {answer.model_dump()}")
print(f"\nInput tokens: {result.input_tokens}, cost: ${result.cost:.6f}")

uses_deep_learning: {'type': 'noul', 'value': 0.99, 'label': None, 'level': None, 'probabilities': None, 'confidence': None}
modality: {'type': 'choice', 'value': 'X-ray', 'label': 'X-ray', 'level': None, 'probabilities': {'CT': 0.0, 'MRI': 0.0, 'X-ray': 1.0, 'ultrasound': 0.0}, 'confidence': 1.0}
validation: {'type': 'score', 'value': 2.0, 'label': 'external', 'level': 2, 'probabilities': {'none': 0.0, 'internal only': 0.0, 'external': 1.0}, 'confidence': 1.0}

Input tokens: 448, cost: $0.000019


Note that the `Noul` answer has no confidence: its probability already says how certain the model is.

## Data

Let's load the same dataset of radiology AI articles used in the title/abstract tutorial, and take 100 of them.

In [5]:
data = pd.read_csv("data.csv").sample(100, random_state=42).reset_index(drop=True)
data[["title", "abstract", "modality", "study_type"]].head()

,title,abstract,modality,study_type
0,"Automatic classification between COVID-19 pneumonia, non-COVID-19 ...",This study aimed to develop and validate computer-aided diagnosis ...,xr,cross-sectional
1,Explaining Deep Features Using Radiologist-Defined Semantic Featur...,Quantitative features are generated from a tumor phenotype by vari...,ct,cross-sectional
2,Augmenting existing deterioration indices with chest radiographs t...,"IMPORTANCE: When hospitals are at capacity, accurate deterioration...",xr,cross-sectional
3,Three-Dimensional Texture Feature Analysis of Pulmonary Nodules in...,To extract texture features of pulmonary nodules from three-dimens...,ct,cross-sectional
4,A novel deep learning-based quantification of serial chest compute...,This study aims to explore and compare a novel deep learning-based...,ct,cross-sectional


We use the same criteria as the title/abstract tutorial. Decision reviewers also accept a list or a numbered dict,
e.g., `{1: "...", 2: "..."}`, and ask one question per criterion.

In [6]:
inclusion_criteria = "The study must involve CT scans. If multiple modalities are involved, CT scans should be among them."
exclusion_criteria = "The study must not include PET scans as one of its modalities."

## Screening with DecisionTitleAbstractReviewer

`DecisionTitleAbstractReviewer` is the decision-model counterpart of `TitleAbstractReviewer`. For each article it
sends one request with:

- a 5-level `evaluation` score that mirrors the LLM reviewer's 1-5 rubric,
- an overall `include` question ("meets ALL inclusion and NONE of the exclusion criteria"),
- one yes/no question per inclusion and exclusion criterion.

In [7]:
jev_reviewer = DecisionTitleAbstractReviewer(
    provider=provider,
    name="Jev",
    inclusion_criteria=inclusion_criteria,
    exclusion_criteria=exclusion_criteria,
)

jev_reviewer.response_format

{'evaluation': int,
 'include_probability': float,
 'confidence': float,
 'criteria': dict,
 'reasoning': str}

In [8]:
workflow = ReviewWorkflow(
    workflow_schema=[
        {
            "round": "A",
            "reviewers": [jev_reviewer],
            "text_inputs": ["title", "abstract"],
        }
    ],
    verbose=False,
)

start = time.time()
screened = asyncio.run(workflow(data))
seconds = time.time() - start
print(f"Screened {len(screened)} articles in {seconds:.1f} s for ${workflow.get_total_cost():.4f}")

['round: A', 'reviewer_name: Jev'] -                     2026-09-27 17:41:37:   0%|          | 0/100 [00:00<?, ?it/s]

['round: A', 'reviewer_name: Jev'] -                     2026-09-27 17:41:37:   2%|▏         | 2/100 [00:00<00:10,  9.40it/s]

['round: A', 'reviewer_name: Jev'] -                     2026-09-27 17:41:37:   3%|▎         | 3/100 [00:00<00:15,  6.13it/s]

['round: A', 'reviewer_name: Jev'] -                     2026-09-27 17:41:37:   7%|▋         | 7/100 [00:00<00:07, 12.17it/s]

['round: A', 'reviewer_name: Jev'] -                     2026-09-27 17:41:37:  12%|█▏        | 12/100 [00:00<00:05, 17.29it/s]

...

['round: A', 'reviewer_name: Jev'] -                     2026-09-27 17:41:37:  94%|█████████▍| 94/100 [00:05<00:00, 18.31it/s]

['round: A', 'reviewer_name: Jev'] -                     2026-09-27 17:41:37:  96%|█████████▌| 96/100 [00:05<00:00, 15.00it/s]

['round: A', 'reviewer_name: Jev'] -                     2026-09-27 17:41:37:  99%|█████████▉| 99/100 [00:06<00:00, 16.19it/s]

['round: A', 'reviewer_name: Jev'] -                     2026-09-27 17:41:37: 100%|██████████| 100/100 [00:06<00:00, 16.33it/s]

Screened 100 articles in 6.1 s for $0.0047


In [9]:
screened[["title", "modality", "round-A_Jev_evaluation", "round-A_Jev_include_probability", "round-A_Jev_confidence"]].head(10)

,title,modality,round-A_Jev_evaluation,round-A_Jev_include_probability,round-A_Jev_confidence
0,"Automatic classification between COVID-19 pneumonia, non-COVID-19 ...",xr,1,0.02,0.99
1,Explaining Deep Features Using Radiologist-Defined Semantic Featur...,ct,3,0.6,0.36
2,Augmenting existing deterioration indices with chest radiographs t...,xr,1,0.03,0.99
3,Three-Dimensional Texture Feature Analysis of Pulmonary Nodules in...,ct,5,0.97,0.99
4,A novel deep learning-based quantification of serial chest compute...,ct,5,0.98,0.99
5,Disease Progression Detection via Deep Sequence Learning of Succes...,xr,1,0.03,0.97
6,Performance improvement of mediastinal lymph node severity detecti...,ct,5,0.94,0.97
7,External validation of a convolutional neural network artificial i...,ct,5,0.95,0.96
8,Improved lung nodule diagnosis accuracy using lung CT images with ...,ct,5,0.96,0.98
9,Artificial intelligence for the detection of COVID-19 pneumonia on...,ct,5,0.97,0.99


The `evaluation` column uses the same 1-5 scale as `TitleAbstractReviewer`, so existing filters keep working. Jev's
probabilities are usually close to 0 or 1:

In [10]:
screened["round-A_Jev_evaluation"].value_counts().sort_index()

round-A_Jev_evaluation
1    39
3     3
5    58
Name: count, dtype: int64

## Inspecting the probabilities

The `criteria` column holds the probability for each criterion, and `reasoning` summarizes them in words.
**The reasoning is generated by LatteReview from the probabilities; Jev does not write it.**

In [11]:
for _, row in screened.head(4).iterrows():
    print(f"Title: {row['title']}")
    print(f"Modality (label): {row['modality']}")
    print(f"Criteria: {row['round-A_Jev_criteria']}")
    print(f"Reasoning: {row['round-A_Jev_reasoning']}")
    print("-" * 100)

Title: Automatic classification between COVID-19 pneumonia, non-COVID-19 pneumonia, and the healthy on chest X-ray image: combination of data augmentation methods
Modality (label): xr
Criteria: {'inclusion': {'The study must involve CT scans. If multiple modalities are involved, CT scans should be among them.': 0.02}, 'exclusion': {'The study must not include PET scans as one of its modalities.': 0.06}}
Reasoning: P(include)=0.02; rated 'absolutely exclude'. Likely fails inclusion 1 (p=0.02: 'The study must involve CT scans. If multiple modalities are…'). No exclusion criterion likely applies (highest p=0.06).
----------------------------------------------------------------------------------------------------
Title: Explaining Deep Features Using Radiologist-Defined Semantic Features and Traditional Quantitative Features
Modality (label): ct
Criteria: {'inclusion': {'The study must involve CT scans. If multiple modalities are involved, CT scans should be among them.': 0.73}, 'exclusion

The full answers (per-level probabilities, confidence, and the answer to every question) are kept under `_answers`
in the output column:

In [12]:
screened.loc[0, "round-A_Jev_output"]["_answers"]["evaluation"]

{'type': 'score',
 'value': 0.01,
 'label': 'absolutely exclude',
 'level': 0,
 'probabilities': {'absolutely exclude': 0.99,
  'better to exclude': 0.01,
  'not sure whether to include or exclude': 0.0,
  'better to include': 0.0,
  'absolutely include': 0.0},
 'confidence': 0.99}

## Scoring with DecisionScoringReviewer

`DecisionScoringReviewer` is the counterpart of `ScoringReviewer`. `score_descriptions` become the answer levels the
model chooses from. `certainty` is the model's confidence x 100, on the same 0-100 scale as `ScoringReviewer`.

In [13]:
validation_reviewer = DecisionScoringReviewer(
    provider=provider,
    name="Validation",
    scoring_task="How strong is the validation of the model or method in this study?",
    scoring_set=[1, 2, 3],
    score_descriptions={
        1: "no validation reported",
        2: "internal validation only (e.g., a held-out split or cross-validation)",
        3: "external validation on data from another institution",
    },
)

workflow = ReviewWorkflow(
    workflow_schema=[{"round": "A", "reviewers": [validation_reviewer], "text_inputs": ["title", "abstract"]}],
    verbose=False,
)
scored = asyncio.run(workflow(data))
scored[["title", "round-A_Validation_score", "round-A_Validation_certainty", "round-A_Validation_probabilities"]].head()

['round: A', 'reviewer_name: Validation'] -                     2026-09-27 17:41:43:   0%|          | 0/100 [00:00<?, ?it/s]

['round: A', 'reviewer_name: Validation'] -                     2026-09-27 17:41:43:   2%|▏         | 2/100 [00:00<00:13,  7.43it/s]

['round: A', 'reviewer_name: Validation'] -                     2026-09-27 17:41:43:   5%|▌         | 5/100 [00:00<00:06, 13.65it/s]

['round: A', 'reviewer_name: Validation'] -                     2026-09-27 17:41:43:   8%|▊         | 8/100 [00:00<00:06, 14.48it/s]

['round: A', 'reviewer_name: Validation'] -                     2026-09-27 17:41:43:  11%|█         | 11/100 [00:00<00:05, 14.84it/s]

...

['round: A', 'reviewer_name: Validation'] -                     2026-09-27 17:41:43:  95%|█████████▌| 95/100 [00:05<00:00, 18.59it/s]

['round: A', 'reviewer_name: Validation'] -                     2026-09-27 17:41:43:  97%|█████████▋| 97/100 [00:05<00:00, 17.27it/s]

['round: A', 'reviewer_name: Validation'] -                     2026-09-27 17:41:43:  99%|█████████▉| 99/100 [00:06<00:00, 16.53it/s]

['round: A', 'reviewer_name: Validation'] -                     2026-09-27 17:41:43: 100%|██████████| 100/100 [00:06<00:00, 16.35it/s]

,title,round-A_Validation_score,round-A_Validation_certainty,round-A_Validation_probabilities
0,"Automatic classification between COVID-19 pneumonia, non-COVID-19 ...",2,100,"{1: 0.0, 2: 1.0, 3: 0.0}"
1,Explaining Deep Features Using Radiologist-Defined Semantic Featur...,1,84,"{1: 0.9, 2: 0.1, 3: 0.0}"
2,Augmenting existing deterioration indices with chest radiographs t...,2,89,"{1: 0.0, 2: 0.93, 3: 0.07}"
3,Three-Dimensional Texture Feature Analysis of Pulmonary Nodules in...,2,56,"{1: 0.13, 2: 0.71, 3: 0.16}"
4,A novel deep learning-based quantification of serial chest compute...,2,28,"{1: 0.48, 2: 0.52, 3: 0.0}"


## Categorical extraction with DecisionReviewer

For your own questions, use the generic `DecisionReviewer`. Each question becomes one output column: the chosen
option for `Choice`, the probability of "yes" for `Noul`, and the expected level for `Score`. Here we extract the
imaging modality and the study type, which the dataset also has as labels, so we can check the answers.
(Free-text extraction still needs an LLM `AbstractionReviewer`.)

In [14]:
extractor = DecisionReviewer(
    provider=provider,
    name="Extractor",
    questions={
        "modality": Choice(
            "What is the main imaging modality of the study?",
            {"ct": "CT (computed tomography)", "xr": "X-ray radiography", "pet": "PET or PET/CT", "mri": "MRI", "us": "ultrasound"},
        ),
        "study_type": Choice(
            "What is the study design?",
            {
                "cross-sectional": "cross-sectional (e.g., a diagnostic accuracy study on existing images)",
                "retrospective cohort": "retrospective cohort",
                "prospective cohort": "prospective cohort",
                "case control": "case-control",
                "RCT": "randomized controlled trial",
            },
        ),
        "deep_learning": Noul("Does the study use deep learning?"),
    },
)

workflow = ReviewWorkflow(
    workflow_schema=[{"round": "A", "reviewers": [extractor], "text_inputs": ["title", "abstract"]}],
    verbose=False,
)
extracted = asyncio.run(workflow(data))
extracted[["title", "modality", "round-A_Extractor_modality", "study_type", "round-A_Extractor_study_type", "round-A_Extractor_deep_learning"]].head(10)

['round: A', 'reviewer_name: Extractor'] -                     2026-09-27 17:41:49:   0%|          | 0/100 [00:00<?, ?it/s]

['round: A', 'reviewer_name: Extractor'] -                     2026-09-27 17:41:49:   2%|▏         | 2/100 [00:00<00:11,  8.25it/s]

['round: A', 'reviewer_name: Extractor'] -                     2026-09-27 17:41:49:   5%|▌         | 5/100 [00:00<00:07, 12.16it/s]

['round: A', 'reviewer_name: Extractor'] -                     2026-09-27 17:41:49:   8%|▊         | 8/100 [00:00<00:06, 14.85it/s]

['round: A', 'reviewer_name: Extractor'] -                     2026-09-27 17:41:49:  11%|█         | 11/100 [00:00<00:05, 15.82it/s]

...

['round: A', 'reviewer_name: Extractor'] -                     2026-09-27 17:41:49:  94%|█████████▍| 94/100 [00:05<00:00, 17.57it/s]

['round: A', 'reviewer_name: Extractor'] -                     2026-09-27 17:41:49:  96%|█████████▌| 96/100 [00:05<00:00, 16.60it/s]

['round: A', 'reviewer_name: Extractor'] -                     2026-09-27 17:41:49:  98%|█████████▊| 98/100 [00:06<00:00, 16.34it/s]

['round: A', 'reviewer_name: Extractor'] -                     2026-09-27 17:41:49: 100%|██████████| 100/100 [00:06<00:00, 16.13it/s]

,title,modality,round-A_Extractor_modality,study_type,round-A_Extractor_study_type,round-A_Extractor_deep_learning
0,"Automatic classification between COVID-19 pneumonia, non-COVID-19 ...",xr,xr,cross-sectional,cross-sectional,0.99
1,Explaining Deep Features Using Radiologist-Defined Semantic Featur...,ct,ct,cross-sectional,cross-sectional,0.99
2,Augmenting existing deterioration indices with chest radiographs t...,xr,xr,cross-sectional,retrospective cohort,0.73
3,Three-Dimensional Texture Feature Analysis of Pulmonary Nodules in...,ct,ct,cross-sectional,cross-sectional,0.03
4,A novel deep learning-based quantification of serial chest compute...,ct,ct,cross-sectional,retrospective cohort,0.99
5,Disease Progression Detection via Deep Sequence Learning of Succes...,xr,xr,retrospective cohort,retrospective cohort,0.99
6,Performance improvement of mediastinal lymph node severity detecti...,ct,ct,cross-sectional,cross-sectional,0.99
7,External validation of a convolutional neural network artificial i...,ct,ct,cross-sectional,cross-sectional,0.98
8,Improved lung nodule diagnosis accuracy using lung CT images with ...,ct,ct,cross-sectional,cross-sectional,0.13
9,Artificial intelligence for the detection of COVID-19 pneumonia on...,ct,ct,cross-sectional,cross-sectional,0.99


In [15]:
for column in ["modality", "study_type"]:
    agreement = (extracted[column] == extracted[f"round-A_Extractor_{column}"]).mean()
    print(f"{column}: Jev matches the label for {agreement:.0%} of articles")

modality: Jev matches the label for 95% of articles
study_type: Jev matches the label for 82% of articles


## Cost, speed, and choosing a threshold

Every reviewer tracks its cost. Jev bills input tokens only, and the state is billed once per request no matter how
many questions it carries.

In [16]:
for reviewer in [jev_reviewer, validation_reviewer, extractor]:
    print(f"{reviewer.name}: {len(reviewer.memory)} articles, ${reviewer.cost_so_far:.5f} "
          f"(${1000 * reviewer.cost_so_far / len(reviewer.memory):.4f} per 1,000 articles)")

Jev: 100 articles, $0.00469 ($0.0469 per 1,000 articles)
Validation: 100 articles, $0.00310 ($0.0310 per 1,000 articles)
Extractor: 100 articles, $0.00381 ($0.0381 per 1,000 articles)


`include_probability` can be thresholded. With labeled data, `suggest_threshold` finds the highest cutoff that
still reaches a target recall. Here, our label is the CT-without-PET rule applied to the dataset's `modality`
column. A cutoff fitted on one dataset and backend does not carry over to another, so fit it on a labeled sample of
your own data.

In [17]:
screened["label"] = screened["modality"].isin(["ct", "xr, ct"]).astype(int)
suggest_threshold(screened, "round-A_Jev_include_probability", "label", target_recall=0.95)

{'threshold': 0.13,
 'recall': 0.95,
 'precision': 0.890625,
 'n_included': 64,
 'wss': 0.30999999999999994}

In [18]:
asyncio.run(provider.aclose())